# Experiment 2: does the anchor underfit?

The anchor ran LightGBM's defaults, which is 100 trees at learning rate 0.1, and
finished in 23.5 seconds on 691,369 rows. That is a strong hint of underfitting. The
top of the public leaderboard is at 0.97102 against the anchor's 0.95594, a gap of
about 0.015 AUC, roughly 23 fold standard deviations. Far too large to be noise.

This notebook changes **one variable: `n_estimators`.** Learning rate stays at the
default 0.1, every other parameter stays at its default, the seed stays 42, and the
folds are constructed identically so they are the same folds row for row.

**No early stopping.** It would let the validation fold choose the tree count, which
makes the CV score optimistic and, worse, makes these numbers incomparable to the
anchor's. The tree count is swept explicitly instead, which costs more compute and
keeps the ledger honest.

The 100-tree level is included as a reproducibility check. It must return the anchor's
0.954947. If it does not, something in this pipeline is non-deterministic and that
matters more than any score in this notebook.

In [1]:
import csv
import time
from datetime import datetime, timezone
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID = "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

# The one variable under test. 100 reproduces the anchor.
N_ESTIMATORS_GRID = [100, 300, 1000, 2000]
ANCHOR_CV = 0.954947

print("lightgbm", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

lightgbm 4.7.0 | pandas 3.0.5 | numpy 2.5.1


In [2]:
def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv locally or under /kaggle/input")


REPO, RAW = locate()
SUB_DIR = REPO / "submissions"
OOF_DIR = REPO / "artifacts" / "oof"
SUB_DIR.mkdir(parents=True, exist_ok=True)
OOF_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
sample = pd.read_csv(RAW / "sample_submission.csv")

FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
for c in CAT_COLS:
    levels = pd.Categorical(pd.concat([train[c], test[c]], ignore_index=True)).categories
    train[c] = pd.Categorical(train[c], categories=levels)
    test[c] = pd.Categorical(test[c], categories=levels)

y = train[TARGET].to_numpy()

assert ID not in FEATURES, "id must never be a feature"
assert TARGET not in FEATURES, "target must never be a feature"
assert not (set(train[ID]) & set(test[ID])), "train and test ids overlap"
assert list(FEATURES) == [c for c in test.columns if c != ID], "train/test feature mismatch"

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.full(len(train), -1, dtype=int)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i
assert (folds >= 0).all(), "every row must land in exactly one fold"

print(f"{len(train):,} train rows, {len(FEATURES)} features, {N_SPLITS} folds")

691,369 train rows, 12 features, 5 folds


## Sweep

One full 5-fold run per tree count. The expensive way round, and the only way the
numbers stay comparable to the anchor.

In [3]:
def run_level(n_estimators):
    oof = np.zeros(len(train), dtype=float)
    test_pred = np.zeros(len(test), dtype=float)
    fold_scores = []
    t0 = time.time()

    for f in range(N_SPLITS):
        tr_m, va_m = folds != f, folds == f
        model = lgb.LGBMClassifier(n_estimators=n_estimators, random_state=SEED, verbose=-1)
        model.fit(train.loc[tr_m, FEATURES], y[tr_m])
        p_va = model.predict_proba(train.loc[va_m, FEATURES])[:, 1]
        oof[va_m] = p_va
        test_pred += model.predict_proba(test[FEATURES])[:, 1] / N_SPLITS
        fold_scores.append(roc_auc_score(y[va_m], p_va))

    return {
        "n_estimators": n_estimators,
        "cv_mean": float(np.mean(fold_scores)),
        "cv_std": float(np.std(fold_scores)),
        "pooled": float(roc_auc_score(y, oof)),
        "secs": time.time() - t0,
        "oof": oof,
        "test_pred": test_pred,
    }


results = []
for n in N_ESTIMATORS_GRID:
    r = run_level(n)
    results.append(r)
    print(f"  n_estimators={n:>5}  cv={r['cv_mean']:.6f} +/- {r['cv_std']:.6f}  "
          f"pooled={r['pooled']:.6f}  {r['secs']:.0f}s")

  n_estimators=  100  cv=0.954947 +/- 0.000645  pooled=0.954944  28s


  n_estimators=  300  cv=0.960605 +/- 0.000688  pooled=0.960604  61s


  n_estimators= 1000  cv=0.962141 +/- 0.000859  pooled=0.962137  209s


  n_estimators= 2000  cv=0.961832 +/- 0.000952  pooled=0.961807  318s


## Reproducibility check

The 100-tree level must reproduce the anchor exactly. Anything else means the pipeline
is not deterministic, and every comparison in the ledger would be built on sand.

In [4]:
repro = next(r for r in results if r["n_estimators"] == 100)
delta = abs(repro["cv_mean"] - ANCHOR_CV)
print(f"anchor CV     : {ANCHOR_CV:.6f}")
print(f"reproduced CV : {repro['cv_mean']:.6f}")
print(f"delta         : {delta:.9f}")
assert delta < 1e-6, (
    f"pipeline is not deterministic: 100 trees gave {repro['cv_mean']:.6f}, "
    f"anchor gave {ANCHOR_CV:.6f}. Stop and diagnose before trusting any comparison."
)
print("\ndeterministic: same config, same number")

anchor CV     : 0.954947
reproduced CV : 0.954947
delta         : 0.000000299

deterministic: same config, same number


## Read the sweep

The question is not which number is biggest. It is whether the improvement is larger
than the fold spread, and whether the curve has flattened or is still climbing.

In [5]:
best = max(results, key=lambda r: r["cv_mean"])
print(f"{'n_est':>6} {'cv':>10} {'sd':>9} {'vs anchor':>11} {'vs sd':>7} {'secs':>6}")
print("-" * 54)
for r in results:
    gain = r["cv_mean"] - ANCHOR_CV
    in_sd = gain / r["cv_std"] if r["cv_std"] else float("nan")
    print(f"{r['n_estimators']:>6} {r['cv_mean']:>10.6f} {r['cv_std']:>9.6f} "
          f"{gain:>+11.6f} {in_sd:>7.1f} {r['secs']:>6.0f}")

print(f"\nbest: n_estimators={best['n_estimators']} at {best['cv_mean']:.6f}")
gain = best["cv_mean"] - ANCHOR_CV
print(f"gain over anchor: {gain:+.6f}, which is {gain / best['cv_std']:.1f} fold sd")
if gain < best["cv_std"]:
    print("\nThat gain is inside one fold standard deviation. Do not believe it "
          "without a seed sweep.")
if best["n_estimators"] == N_ESTIMATORS_GRID[-1]:
    print("\nBest is at the top of the grid, so the curve has not turned over yet. "
          "Extend the grid before concluding anything about the optimum.")

 n_est         cv        sd   vs anchor   vs sd   secs
------------------------------------------------------
   100   0.954947  0.000645   -0.000000    -0.0     28
   300   0.960605  0.000688   +0.005658     8.2     61
  1000   0.962141  0.000859   +0.007194     8.4    209
  2000   0.961832  0.000952   +0.006885     7.2    318

best: n_estimators=1000 at 0.962141
gain over anchor: +0.007194, which is 8.4 fold sd


## Save and log

One ledger row per level, including the 100-tree reproducibility re-run. The rule in
`CLAUDE.md` is that every run appends a row, and bending it for tidiness is how ledgers
stop being trustworthy.

In [6]:
LEDGER = REPO / "experiments.csv"
COLUMNS = ["id", "utc", "name", "cv_mean", "cv_std", "folds",
           "lb_public", "lb_private", "submitted", "notes"]

rows = []
if LEDGER.exists():
    with LEDGER.open(newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
next_id = max((int(r["id"]) for r in rows), default=0) + 1

for r in results:
    n = r["n_estimators"]
    tag = f"lgbm_trees{n}_seed{SEED}"
    np.save(OOF_DIR / f"{tag}.npy", r["oof"])
    sub = sample.copy()
    sub[TARGET] = r["test_pred"]
    sub.to_csv(SUB_DIR / f"{tag}.csv", index=False)

    note = f"n_estimators={n}, lr default 0.1, no early stopping, all else anchor"
    if n == 100:
        note = "reproducibility re-run of exp 1, same config, not a new idea"

    rows.append({
        "id": str(next_id), "utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
        "name": f"lgbm_trees{n}", "cv_mean": f"{r['cv_mean']:.6f}",
        "cv_std": f"{r['cv_std']:.6f}", "folds": str(N_SPLITS),
        "lb_public": "", "lb_private": "", "submitted": "no", "notes": note,
    })
    next_id += 1

with LEDGER.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=COLUMNS)
    w.writeheader()
    w.writerows({c: r.get(c, "") for c in COLUMNS} for r in rows)

print("ledger now:")
pd.read_csv(LEDGER)

ledger now:


,id,utc,name,cv_mean,cv_std,folds,lb_public,lb_private,submitted,notes
0,1,2026-08-04 02:15,lgbm_default_anchor,0.954947,0.000645,5,0.95594,NaN,yes,"untuned lgbm defaults, raw features, native ca..."
1,2,2026-08-04 05:29,lgbm_trees100,0.954947,0.000645,5,NaN,NaN,no,"reproducibility re-run of exp 1, same config, ..."
2,3,2026-08-04 05:29,lgbm_trees300,0.960605,0.000688,5,NaN,NaN,no,"n_estimators=300, lr default 0.1, no early sto..."
3,4,2026-08-04 05:29,lgbm_trees1000,0.962141,0.000859,5,NaN,NaN,no,"n_estimators=1000, lr default 0.1, no early st..."
4,5,2026-08-04 05:29,lgbm_trees2000,0.961832,0.000952,5,NaN,NaN,no,"n_estimators=2000, lr default 0.1, no early st..."


## Next

Submit only the best level, and only if its gain clears the fold spread. Submitting
every level burns the 10-per-day budget to buy leaderboard readings of configurations
that CV has already ranked.

```
kaggle competitions submit -c playground-series-s6e8 -f submissions/<best>.csv -m "<note>"
```